<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 12th exercise: <font color="#C70039">Q-table learning on a deterministic graph</font>

* Course: AML  
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Date: 03.09.2026

---

**GENERAL NOTE 1**: 
Please make sure you are reading the entire notebook, since it contains a lot of information on your tasks (e.g. regarding the set of certain parameters or a specific computational trick), and the written mark downs as well as comments contain a lot of information on how things work together as a whole. 

**GENERAL NOTE 2**: 
* Please, when commenting source code, just use English language only. 
* When describing an observation please use English language, too.
* This applies to all exercises throughout this course.

---------------------------------

### <font color="FFC300">LEARNING OBJECTIVES</font>:

After this exercise, you can represent a finite decision problem as states, actions, and rewards; implement the tabular Q-learning update; explain exploration versus exploitation; and derive a route from a learned Q-table.

### <font color="green">SAMPLE SOLUTION</font>:

This notebook is a completed copy of the corresponding exercise Ex12. It shows one valid implementation and concise explanations of the design choices. The explanations focus on why each step is needed.

### <font color="ce33ff">DESCRIPTION</font>:

An agent moves between nine locations. A positive entry in the adjacency matrix represents a permitted move. The agent receives a reward of **10** when it enters the destination location `L4`; all other permitted moves have reward **0**. Episodes therefore end when `L4` is reached.

This deliberately small and deterministic problem isolates the Q-learning mechanism before we use a Gymnasium environment in Exercise 13.

In [ ]:
import numpy as np

# A fixed generator makes experiments reproducible.
rng = np.random.default_rng(1)

location_to_state = {f"L{index}": index - 1 for index in range(1, 10)}
state_to_location = {state: location for location, state in location_to_state.items()}
goal_location = "L4"
goal_state = location_to_state[goal_location]

# Rows are current states and columns are possible next states.
adjacency = np.array([
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 1, 0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 1, 0],
], dtype=int)

rewards = np.zeros_like(adjacency, dtype=float)
rewards[:, goal_state] = 10.0
rewards[adjacency == 0] = np.nan

print("State mapping:", location_to_state)
print("Destination:", goal_location)

## Hyperparameters and Q-table

`alpha` controls how strongly new experience changes a Q-value. `gamma` discounts future reward. Epsilon starts high to encourage exploration and decreases after each episode.

Run the next cell once before completing the two functions below.

In [ ]:
num_episodes = 1_000
max_steps_per_episode = 20
alpha = 0.7
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

# The table is zero-initialised. Action selection must still use reachable actions only.
q_table = np.zeros_like(rewards, dtype=float)

## <font color="FFC300">Task 1</font> — action selection

Complete `choose_action`. With probability epsilon it must choose a random *reachable* action. Otherwise it must choose the reachable action with the highest Q-value. Do not select an action from a zero entry of `adjacency`.

Explain in a Markdown cell why choosing from all nine actions would be incorrect.

### Sample answer — Task 1

Only reachable actions represent valid state transitions. Sampling from all nine destination indices would allow the agent to learn values for transitions that do not exist in the graph. The epsilon-greedy policy therefore first obtains `reachable_actions`, then explores or exploits only within that set.

In [ ]:
def choose_action(state, q_table, epsilon):
    reachable_actions = np.flatnonzero(adjacency[state])

    # Explore only transitions that are permitted from the current state.
    if rng.random() < epsilon:
        return int(rng.choice(reachable_actions))

    # Exploit the largest Q-value among the permitted actions.
    reachable_values = q_table[state, reachable_actions]
    return int(reachable_actions[np.argmax(reachable_values)])

## <font color="FFC300">Task 2</font> — Q-learning update

Complete the update according to

$$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)].$$

For a terminal transition, the future value is zero. This is important because no action follows once the goal has been reached.

### Sample answer — Task 2

The update uses the temporal-difference error: immediate reward plus the discounted best successor value minus the current estimate. For a terminal transition the successor value is zero because no next decision exists; bootstrapping beyond the goal would invent future reward.

In [ ]:
def update_q_value(state, action, next_state, reward, terminal):
    # A terminal state has no successor value to bootstrap from.
    future_value = 0.0 if terminal else np.max(q_table[next_state])
    temporal_difference = reward + gamma * future_value - q_table[state, action]
    q_table[state, action] += alpha * temporal_difference

## Training

After completing Tasks 1 and 2, run this cell. Each episode begins in a randomly selected non-goal state. The printed success rate should converge to one because the graph is deterministic and every sampled start state can reach `L4`.

In [ ]:
episode_lengths = []

for episode in range(num_episodes):
    state = int(rng.integers(len(location_to_state)))
    while state == goal_state:
        state = int(rng.integers(len(location_to_state)))

    for step in range(max_steps_per_episode):
        action = choose_action(state, q_table, epsilon)
        next_state = action
        terminal = next_state == goal_state
        reward = rewards[state, action]
        update_q_value(state, action, next_state, reward, terminal)
        state = next_state

        if terminal:
            episode_lengths.append(step + 1)
            break
    else:
        episode_lengths.append(np.nan)

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print(f"Completed episodes: {np.isfinite(episode_lengths).mean():.1%}")
print(f"Mean successful episode length: {np.nanmean(episode_lengths):.2f}")

## <font color="FFC300">Task 3</font> — derive and evaluate a policy

Complete the policy function and use it to retrieve a route from `L9` to `L4`. Then change **one** hyperparameter at a time and record the effect on convergence and the learned route. Test at least `alpha`, `gamma`, initial epsilon, and epsilon decay.

Finally, alter one connection in `adjacency`. State your prediction before training again and explain the observed policy.

### Sample answer — Task 3

After convergence, the greedy route from `L9` is `L9 → L8 → L7 → L4`. A disciplined test plan changes one of `alpha`, `gamma`, initial epsilon, or epsilon decay while keeping the seed and all remaining parameters constant. Changing an edge changes the available transitions, so the learned policy must be trained again rather than inferred from the old Q-table.

In [ ]:
def greedy_route(start_location, destination_location):
    route = [start_location]
    current_state = location_to_state[start_location]
    destination_state = location_to_state[destination_location]

    for _ in range(len(location_to_state) * 2):
        if current_state == destination_state:
            return route

        reachable_actions = np.flatnonzero(adjacency[current_state])
        reachable_values = q_table[current_state, reachable_actions]
        current_state = int(reachable_actions[np.argmax(reachable_values)])
        route.append(state_to_location[current_state])

    raise RuntimeError("The greedy policy did not reach the destination.")

print("Learned Q-table:")
print(np.round(q_table, 2))
print("Greedy route from L9 to L4:", greedy_route("L9", "L4"))

## <font color="FFC300">Task 4</font> — Reflection

In your own words, answer the following:

1. Why is this decision problem deterministic?
2. Which values encode the goal-directed policy in the Q-table?
3. Why must the set of available actions depend on the current state?
4. What behaviour would you expect if epsilon never decayed?

### Sample answer — Task 4

1. The successor state and reward are fixed by the chosen valid transition.
2. Larger values on actions that move toward `L4` encode the goal-directed policy.
3. Available actions depend on the current graph node; an edge from another node is not necessarily available here.
4. If epsilon never decayed, the agent would keep taking random actions and the observed performance would remain below that of its greedy policy.